In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import pyarrow.parquet as pq
LN2 = np.log(2.0)

def lin(x):
    return np.exp2(x) - 1.0

files = sorted(glob.glob(f"{SLIM_DIR}/*_slim.parquet"))
parts = []
for f in files:
    roi = os.path.basename(f).replace("_slim.parquet", "")
    names = pq.ParquetFile(f).schema.names
    genes = [g for g in GRMS if g in names]
    d = pd.read_parquet(f, columns=["class", "subclass"] + genes)
    d = d[[is_neuron_class(c) for c in d["class"]]].copy()
    for g in genes:
        d[g] = d[g].astype("float32")
    d.insert(0, "roi", roi)
    parts.append(d)
    print(roi, len(d), "neurons", flush=True)
df = pd.concat(parts, ignore_index=True)
for g in GRMS:
    if g not in df.columns:
        df[g] = np.float32(0)
df[GRMS] = df[GRMS].fillna(0)
for c in ("roi", "class", "subclass"):
    df[c] = df[c].astype("category")
thr = {g: float(np.median(df[g][df[g] > 0])) if (df[g] > 0).any() else np.inf for g in GRMS}
print("per-receptor positivity threshold (median log2 among detected neurons):", {g: round(v, 2) for g, v in thr.items()})

rng = np.random.default_rng(0)
cls_n = df["class"].value_counts()
classes = sorted([c for c in cls_n.index if cls_n[c] >= 2000], key=class_num)
sub_means = df.groupby(["class", "subclass"], observed=True)[GRMS].mean()
sub_n = df.groupby(["class", "subclass"], observed=True).size()
fig, axes = plt.subplots(len(GRMS), 1, figsize=(max(13, 0.62 * len(classes)), 2.3 * len(GRMS)), sharex=True)
for ax, g in zip(axes, GRMS):
    data, pos = [], []
    for i, c in enumerate(classes):
        v = df.loc[df["class"] == c, g].to_numpy()
        if len(v) > 4000:
            v = rng.choice(v, 4000, replace=False)
        data.append(v)
        pos.append(i)
    vp = ax.violinplot(data, positions=pos, widths=0.85, showextrema=False)
    for b, c in zip(vp["bodies"], classes):
        b.set_facecolor(class_color(c))
        b.set_edgecolor("k")
        b.set_linewidth(0.3)
        b.set_alpha(0.85)
    ax.set_ylabel(f"{g}\nlog2 expr.", fontsize=10)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[-1].set_xticks(range(len(classes)))
axes[-1].set_xticklabels(classes, rotation=75, ha="right", fontsize=9)
fig.suptitle("Whole mouse brain (Allen WMB, 10x): Grm1-8 expression per neuronal class\nviolin = distribution of per-cell log2(CPM+1) across all neurons of the class (color = class)", fontsize=12, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{FIG_DIR}/ALL_wb_violin_scatter.png", dpi=150, bbox_inches="tight")
plt.show()